In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.1 Base vs instruction models

A **base** model only continues text, it does not answer questions. Instruction
following is *added* afterward by post-training (this unit). A key correction:
**context is not memory**, each call is stateless, the model only knows what is in
the prompt right now, not past conversations.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
out = generate(model, tok, "Question: what is your name? Answer:",
               max_new_tokens=40, seed=0)
print(out)

The base model just continues the text in the style of its training data, it does
not 'answer'. The rest of u7 is about teaching it to follow instructions and human
preferences.

In [ ]:
assert len(out) > len("Question: what is your name? Answer:")